# HIEROGLYPH NLP PIPELINE v15 — GRADUATION FINAL
### Gardiner Codes → Determinative-Aware Assembly → Multi-Query RAG → Seed-X Translation
#### v15 Fixes over v14:
- **FIX A**: Dictionary loading now expands compound words (ḫrp-ḥr-ꞽb) → morpheme entries + compound keys
- **FIX B**: Word Assembly Engine now finds compound+morpheme matches via expanded vocab
- **FIX C**: `hybrid_search()` now runs Multi-Query (joined / spaced / Gardiner phonetics) + fuses with RRF
- **FIX D**: Trie keys cache is version-tracked — auto-refreshes when vocab grows
- **FIX E**: Edit distance is now dynamic based on token length (short tokens → strict, long → lenient)
- **FIX F**: `german_to_english_strict()` is now cached (in-memory + disk) — ~35s saved per repeat call
- **FIX G**: NGROK_TOKEN loaded from env var — no secrets in source code


## CELL 0 - Install Dependencies

In [ ]:
# import subprocess, sys

# def pip(*pkgs):
#     result = subprocess.run(
#         [sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs),
#         capture_output=True, text=True
#     )
#     if result.returncode != 0:
#         print(result.stderr[-300:])

# pip('numpy==1.26.4')
# pip('protobuf==3.20.3')
# pip('transformers==4.44.2', 'sentencepiece', 'accelerate', 'safetensors==0.4.3', 'scipy')
# pip('flask', 'flask-cors', 'pyngrok')
# pip('faiss-cpu', 'sentence-transformers', 'datasets')
# pip('rapidfuzz')
# pip('rank-bm25')
# pip('rouge-score')
# pip('sacrebleu')
# pip('nltk')
# pip('scikit-learn')
# subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)
# print('✅ All dependencies installed')

## CELL 1 - Paths & Config (Auto-detect: Kaggle / Colab / Local)

In [2]:
import os

def _detect_env():
    if os.path.exists('/kaggle/input'):
        return 'kaggle'
    try:
        import google.colab
        return 'colab'
    except ImportError:
        pass
    return 'local'

ENV = _detect_env()
print(f'🖥️  Environment detected: {ENV.upper()}')

if ENV == 'kaggle':
    _DATA_ROOT = '/kaggle/input/datasets/mo3azkhaled/new-heiro3'
    _WORK_ROOT = '/kaggle/working'
elif ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    _DATA_ROOT = '/content/drive/MyDrive/hieroglyph_data'
    _WORK_ROOT = '/content'
else:
    _BASE_DIR  = os.path.dirname(os.path.abspath(__file__))
    _DATA_ROOT = os.path.join(_BASE_DIR, 'data')
    _WORK_ROOT = os.path.join(_BASE_DIR, 'outputs')
    os.makedirs(_WORK_ROOT, exist_ok=True)

GARDINER_PATH      = os.path.join(_DATA_ROOT, 'gardiner_updated.csv')
DICT_WORD_PATH     = os.path.join(_DATA_ROOT, 'egyptian_dictionary.csv')
WORD_DICT_NEW_PATH = os.path.join(_DATA_ROOT, 'egyptian_word_dictionary.csv')
INTENTION_PATH     = os.path.join(_DATA_ROOT, 'intention_dataset.csv')

FAISS_INDEX_PATH = os.path.join(_WORK_ROOT, 'bbaw_faiss.index')
FAISS_META_PATH  = os.path.join(_WORK_ROOT, 'bbaw_meta.pkl')
BM25_CACHE_PATH  = os.path.join(_WORK_ROOT, 'bbaw_bm25.pkl')
TEST_SET_PATH    = os.path.join(_WORK_ROOT, 'tla_test_set.csv')

RRF_K                = 60
HYBRID_ALPHA         = 0.6
RRF_SCORE_MAX        = 1.0 / (RRF_K + 1)       # ≈ 0.0164
SIMILARITY_THRESHOLD = RRF_SCORE_MAX * 0.50    # ≈ 0.0082

TOP_K_RAG         = 55
TRAIN_SPLIT       = 0.99
MAX_PROMPT_TOKENS = 3500

print(f'\n⚙️  RRF config:')
print(f'   k={RRF_K}, alpha={HYBRID_ALPHA}')
print(f'   Max possible RRF score : {RRF_SCORE_MAX:.4f}')
print(f'   SIMILARITY_THRESHOLD   : {SIMILARITY_THRESHOLD:.4f}  ← ~50% of max')

print('\n📂 Checking dataset files:\n')
for label, path in [
    ('Gardiner Sign List', GARDINER_PATH),
    ('Word Dictionary',    DICT_WORD_PATH),
    ('Word Dict (New)',    WORD_DICT_NEW_PATH),
    ('Intention Dataset',  INTENTION_PATH),
]:
    status = '✅ FOUND' if os.path.exists(path) else '❌ NOT FOUND'
    print(f'{status} → {label}')
    print(f'   {path}\n')

print(f'📁 Work dir : {_WORK_ROOT}')
print('✅ Config ready')


🖥️  Environment detected: KAGGLE

⚙️  RRF config:
   k=60, alpha=0.6
   Max possible RRF score : 0.0164
   SIMILARITY_THRESHOLD   : 0.0082  ← ~50% of max

📂 Checking dataset files:

✅ FOUND → Gardiner Sign List
   /kaggle/input/datasets/mo3azkhaled/new-heiro3/gardiner_updated.csv

✅ FOUND → Word Dictionary
   /kaggle/input/datasets/mo3azkhaled/new-heiro3/egyptian_dictionary.csv

✅ FOUND → Word Dict (New)
   /kaggle/input/datasets/mo3azkhaled/new-heiro3/egyptian_word_dictionary.csv

✅ FOUND → Intention Dataset
   /kaggle/input/datasets/mo3azkhaled/new-heiro3/intention_dataset.csv

📁 Work dir : /kaggle/working
✅ Config ready


## CELL 2 - Gardiner Sign List V10: determinative detection

In [3]:
import pandas as pd

df_g = pd.read_csv(GARDINER_PATH)
GARDINER_MAP = {}
DETERMINATIVE_CODES = set()

for _, row in df_g.iterrows():
    code_key = str(row['code']).strip().lower()
    if not code_key or code_key == 'nan':
        continue

    phonetic     = str(row.get('phonetic', '')).strip()     if pd.notna(row.get('phonetic'))     else ''
    phonetic_tla = str(row.get('phonetic_tla', '')).strip() if pd.notna(row.get('phonetic_tla')) else ''
    meaning      = str(row.get('meaning', '')).strip()      if pd.notna(row.get('meaning'))       else ''
    unicode_val  = str(row.get('unicode', '')).strip()      if pd.notna(row.get('unicode'))       else ''

    is_det = (phonetic == '' and phonetic_tla == '')
    if is_det:
        DETERMINATIVE_CODES.add(code_key)

    GARDINER_MAP[code_key] = {
        'phonetic'        : phonetic,
        'phonetic_tla'    : phonetic_tla,
        'meaning'         : meaning,
        'unicode'         : unicode_val,
        'is_determinative': is_det,
    }

print(f'Gardiner Sign List loaded: {len(GARDINER_MAP)} signs')
print(f'  Phonetic signs   : {len(GARDINER_MAP) - len(DETERMINATIVE_CODES)}')
print(f'  Determinatives   : {len(DETERMINATIVE_CODES)} (silent word-boundary markers)')

d57 = GARDINER_MAP.get('d57', {})
print(f'\nD57 check: phonetic="{d57.get("phonetic","")}" | is_det={d57.get("is_determinative")} | meaning="{d57.get("meaning","")}"')
print('EXPECTED: phonetic="" | is_det=True  (D57 is a determinative - leg with knife)')


Gardiner Sign List loaded: 819 signs
  Phonetic signs   : 563
  Determinatives   : 256 (silent word-boundary markers)

D57 check: phonetic="" | is_det=True | meaning="Leg with knife"
EXPECTED: phonetic="" | is_det=True  (D57 is a determinative - leg with knife)


## CELL 3 - Word Dictionary (V12 — boundary detection + meanings)

In [4]:
import pandas as pd
import re as _re_dict
import unicodedata as _ud_dict

df_word_dict = pd.read_csv(DICT_WORD_PATH)
WORD_DICT    = {}
for _, row in df_word_dict.iterrows():
    transliteration = str(row['transliteration']).strip().lower()
    english         = str(row['english']).strip() if pd.notna(row['english']) else ''
    if transliteration and transliteration != 'nan':
        WORD_DICT[transliteration] = english
print(f'✅ Word dictionary (meanings) loaded: {len(WORD_DICT)} entries')

WORD_VOCAB      = {}
WORD_VOCAB_NORM = set()

# ── FIX A: Morpheme Expansion helpers ─────────────────────────────────────────
_CHAR_MAP_DICT = {
    '\ua723':'a','\ua725':'a','\ua7bd':'i',
    'y':'y','w':'w','b':'b','p':'p','f':'f','m':'m','n':'n','r':'r','h':'h',
    '\u1e25':'h','\u1e2b':'kh','\u1e96':'kh','s':'s','\u0161':'sh',
    '\u1e33':'q','q':'q','k':'k','g':'g','t':'t','\u1e6f':'tj',
    'd':'d','\u1e0f':'dj','\u1e6d':'t','\u1e0d':'d','\u1e63':'s','\u1e93':'z','i':'i',
}

def _norm_key_dict(word: str) -> str:
    """Normalize TLA word → ASCII key (same logic as normalize_tla but no - stripping)."""
    word = _re_dict.sub(r'[()]', '', word)
    word = _ud_dict.normalize('NFC', word)
    word = ''.join(c for c in word if not _ud_dict.combining(c))
    word = word.lower()
    for egy, lat in _CHAR_MAP_DICT.items():
        word = word.replace(egy.lower(), lat)
    return word.strip()

def _split_morphemes_dict(tla_word: str) -> list:
    """Split 'ḫrp-ḥr(.ꞽ)-ꞽb' → ['ḫrp', 'ḥr', 'ꞽb']"""
    parts = tla_word.split('-')
    result = []
    for p in parts:
        p = _re_dict.sub(r'\(.*?\)', '', p)
        p = _re_dict.sub(r'=\S+$', '', p)
        p = _re_dict.sub(r'\.\S+$', '', p)
        p = p.strip()
        if p:
            result.append(p)
    return result

if os.path.exists(WORD_DICT_NEW_PATH):
    df_word_new = pd.read_csv(WORD_DICT_NEW_PATH)
    expected_cols = {'normalized', 'original', 'source', 'word_length'}
    actual_cols   = set(df_word_new.columns.str.lower())
    if not expected_cols.issubset(actual_cols):
        print(f'⚠️  Warning: expected columns {expected_cols}, got {set(df_word_new.columns)}')
    df_word_new.columns = df_word_new.columns.str.lower()

    morpheme_count = 0
    for _, row in df_word_new.iterrows():
        normalized = str(row.get('normalized', '')).strip().lower()
        original   = str(row.get('original',   '')).strip() if pd.notna(row.get('original'))   else normalized
        source     = str(row.get('source',      '')).strip() if pd.notna(row.get('source'))     else ''
        try:
            word_len = int(row.get('word_length', 0))
        except (ValueError, TypeError):
            word_len = len(normalized)
        if not normalized or normalized == 'nan':
            continue

        # Entry 1: الكلمة كاملة (زي كان)
        WORD_VOCAB[normalized] = {'original': original, 'source': source, 'word_length': word_len}
        WORD_VOCAB_NORM.add(normalized)

        # FIX A: لو الكلمة مركبة (فيها -)، خزّن morphemes + compound key منفصلاً
        if '-' in original:
            morphemes = _split_morphemes_dict(original)
            for m in morphemes:
                m_norm = _norm_key_dict(m).lower()
                if m_norm and len(m_norm) >= 1 and m_norm not in WORD_VOCAB:
                    WORD_VOCAB[m_norm] = {
                        'original': m, 'source': f'morpheme_from:{original}',
                        'word_length': len(m_norm),
                    }
                    WORD_VOCAB_NORM.add(m_norm)
                    morpheme_count += 1

            # compound key بدون - → يتطابق مع output assemble_words()
            compound_norm = _norm_key_dict(original.replace('-', '')).lower()
            if compound_norm and compound_norm not in WORD_VOCAB:
                WORD_VOCAB[compound_norm] = {
                    'original': original, 'source': 'compound_key',
                    'word_length': word_len,
                }
                WORD_VOCAB_NORM.add(compound_norm)
                morpheme_count += 1

    print(f'✅ Word vocabulary v15 loaded: {len(WORD_VOCAB)} entries')
    print(f'   Morpheme expansions added : {morpheme_count} extra entries')
    sources = df_word_new['source'].value_counts() if 'source' in df_word_new.columns else {}
    print(f'   Sources: {dict(sources.head(5))}')
else:
    print(f'⚠️  egyptian_word_dictionary.csv not found — using legacy fallback')
    for k in WORD_DICT:
        WORD_VOCAB[k] = {'original': k, 'source': 'legacy', 'word_length': len(k)}
    WORD_VOCAB_NORM = set(WORD_VOCAB.keys())

COMBINED_VOCAB = {}
for norm_form in WORD_VOCAB:
    COMBINED_VOCAB[norm_form] = WORD_DICT.get(norm_form, '')
for trans, meaning in WORD_DICT.items():
    if trans not in COMBINED_VOCAB:
        COMBINED_VOCAB[trans] = meaning

print(f'\n📚 Combined vocabulary for Trie: {len(COMBINED_VOCAB)} entries')
print(f'   With English meanings : {sum(1 for v in COMBINED_VOCAB.values() if v)} entries')
print(f'   Boundary-only (no EN) : {sum(1 for v in COMBINED_VOCAB.values() if not v)} entries')


✅ Word dictionary (meanings) loaded: 1274 entries
✅ Word vocabulary (boundary detection) loaded: 31032 entries
   Sources: {'bbaw': 29730, 'tla': 1303}

📚 Combined vocabulary for Trie: 31570 entries
   With English meanings : 1274 entries
   Boundary-only (no EN) : 30296 entries


## CELL 3.5 — Word Assembly Engine v11 (Trie + DP + Fuzzy + Scoring)

In [5]:
from __future__ import annotations
import functools
from typing import Any

MAX_WINDOW      = 5
TOP_N_RESULTS   = 3
LEN_BONUS_COEFF = 0.3

# FIX E: Dynamic edit distance
def _get_max_edit_dist(token_len: int) -> int:
    if token_len <= 2: return 0
    if token_len <= 4: return 1
    return 2

class TrieNode:
    __slots__ = ('children', 'is_end', 'value')
    def __init__(self):
        self.children: dict[str, 'TrieNode'] = {}
        self.is_end  : bool                  = False
        self.value   : str                   = ''

class TrieIndex:
    def __init__(self):
        self.root   = TrieNode()
        self._vocab_size = 0
    def insert(self, key: str, value: str) -> None:
        node = self.root
        for ch in key:
            node = node.children.setdefault(ch, TrieNode())
        node.is_end = True
        node.value  = value
    def search(self, key: str):
        node = self.root
        for ch in key:
            if ch not in node.children: return None
            node = node.children[ch]
        return node.value if node.is_end else None
    def has_prefix(self, prefix: str) -> bool:
        node = self.root
        for ch in prefix:
            if ch not in node.children: return False
            node = node.children[ch]
        return True
    def all_keys(self) -> list[str]:
        result: list[str] = []
        def _dfs(node, path):
            if node.is_end: result.append(''.join(path))
            for ch, child in node.children.items():
                path.append(ch); _dfs(child, path); path.pop()
        _dfs(self.root, [])
        return result

WORD_TRIE = TrieIndex()
for _k, _v in COMBINED_VOCAB.items():
    WORD_TRIE.insert(_k.lower(), _v)
print(f'✅  TrieIndex built: {len(COMBINED_VOCAB):,} entries')

@functools.lru_cache(maxsize=8192)
def levenshtein(s: str, t: str) -> int:
    m, n = len(s), len(t)
    if m == 0: return n
    if n == 0: return m
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev = dp[0]; dp[0] = i
        for j in range(1, n + 1):
            temp = dp[j]
            cost = 0 if s[i-1] == t[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev + cost)
            prev  = temp
    return dp[n]

# FIX D: version-tracked cache
_TRIE_KEYS_CACHE: list[str] = []
_TRIE_KEYS_VOCAB_SIZE: int  = 0

def _refresh_trie_keys_cache():
    global _TRIE_KEYS_CACHE, _TRIE_KEYS_VOCAB_SIZE
    current_size = len(COMBINED_VOCAB)
    if current_size != _TRIE_KEYS_VOCAB_SIZE:
        _TRIE_KEYS_CACHE      = WORD_TRIE.all_keys()
        _TRIE_KEYS_VOCAB_SIZE = current_size
        print(f'   Trie keys cache refreshed: {len(_TRIE_KEYS_CACHE):,} keys')
    else:
        print(f'   Trie keys cache up-to-date: {len(_TRIE_KEYS_CACHE):,} keys (no change)')
_refresh_trie_keys_cache()

def find_candidates(token: str) -> list[dict[str, Any]]:
    t = token.lower().strip()
    results: list[dict] = []
    meaning = WORD_TRIE.search(t)
    if meaning is not None:
        vocab_info = WORD_VOCAB.get(t, {})
        results.append({'key': t, 'meaning': meaning, 'match_type': 'exact',
                         'distance': 0, 'word_length': vocab_info.get('word_length', len(t))})
        return results
    t_len         = len(t)
    max_edit_dist = _get_max_edit_dist(t_len)
    if max_edit_dist == 0:
        return []
    for key in _TRIE_KEYS_CACHE:
        vocab_info   = WORD_VOCAB.get(key, {})
        expected_len = vocab_info.get('word_length', len(key))
        if abs(len(key) - t_len) > max_edit_dist: continue
        if expected_len == 1 and t_len > 3: continue
        dist = levenshtein(t, key)
        if dist <= max_edit_dist:
            meaning_f = WORD_TRIE.search(key) or ''
            results.append({'key': key, 'meaning': meaning_f, 'match_type': 'fuzzy',
                             'distance': dist, 'word_length': expected_len})
    results.sort(key=lambda x: (x['distance'], 0 if x['meaning'] else 1))
    return results

def score_sequence(segments: list[dict]) -> float:
    total = 0.0
    for seg in segments:
        mt = seg.get('match_type', 'unknown')
        w  = seg.get('word', '')
        if mt == 'exact':   total += 3.0 + 0.2 * len(w)
        elif mt == 'fuzzy': total += 1.0 + 0.1 * len(w)
        else:               total -= 2.0
    return round(total, 4)

def assemble_words_v11(tokens: list[str]) -> list[list[dict]]:
    n = len(tokens)
    if n == 0: return []
    greedy_segs = []; all_exact = True
    for tok in tokens:
        cands = find_candidates(tok)
        if cands and cands[0]['match_type'] == 'exact':
            best = cands[0]
            greedy_segs.append({'word': tok, 'match_type': 'exact', 'distance': 0,
                                 'matched_key': best['key'], 'meaning': best['meaning']})
        else:
            all_exact = False; break
    memo: dict[int, list] = {}
    def _dfs(pos):
        if pos == n: return [(0.0, [])]
        if pos in memo: return memo[pos]
        hypotheses = []
        for w in range(1, min(MAX_WINDOW, n - pos) + 1):
            merged = ''.join(tokens[pos : pos + w])
            cands  = find_candidates(merged)
            if cands:
                bc  = cands[0]
                seg = {'word': merged, 'match_type': bc['match_type'],
                        'distance': bc['distance'], 'matched_key': bc['key'],
                        'meaning': bc['meaning']}
            else:
                seg = {'word': merged, 'match_type': 'unknown', 'distance': 99,
                        'matched_key': merged, 'meaning': ''}
            for sub_score, sub_segs in _dfs(pos + w):
                full_segs  = [seg] + sub_segs
                full_score = score_sequence(full_segs)
                hypotheses.append((full_score, full_segs))
        hypotheses.sort(key=lambda x: x[0], reverse=True)
        memo[pos] = hypotheses[: TOP_N_RESULTS * 4]
        return memo[pos]
    all_hyps = _dfs(0)
    if all_exact and greedy_segs:
        greedy_score = score_sequence(greedy_segs)
        all_hyps = [(s, segs) for s, segs in all_hyps
                    if [x['word'] for x in segs] != [x['word'] for x in greedy_segs]]
        all_hyps.insert(0, (greedy_score, greedy_segs))
    all_hyps.sort(key=lambda x: x[0], reverse=True)
    return [segs for _, segs in all_hyps[: TOP_N_RESULTS]]

def word_assembly_result(tokens: list[str]) -> dict:
    if not tokens:
        return {'best_segmentation': [], 'best_metadata': [],
                'alternatives': [], 'best_score': 0.0, 'english_hint': ''}
    clean_tokens = [t.lower().strip() for t in tokens if t.strip()]
    all_hyps     = assemble_words_v11(clean_tokens)
    if not all_hyps:
        all_hyps = [[{'word': t, 'match_type': 'unknown', 'distance': 99,
                       'matched_key': t, 'meaning': ''} for t in clean_tokens]]
    best_segs  = all_hyps[0]
    best_words = [s['word'] for s in best_segs]
    best_meta  = [{'word': s['word'], 'type': s['match_type'], 'meaning': s['meaning']}
                   for s in best_segs]
    alternatives = [[s['word'] for s in hyp] for hyp in all_hyps[1:]]
    meanings     = [s['meaning'] for s in best_segs if s['meaning']]
    english_hint = ' / '.join(meanings) if meanings else ''
    return {'best_segmentation': best_words, 'best_metadata': best_meta,
            'alternatives': alternatives, 'best_score': score_sequence(best_segs),
            'english_hint': english_hint}

print('\n✅  Word Assembly Engine v15 ready (dynamic edit dist + morpheme-aware)')


✅  TrieIndex built: 31,570 entries
   Trie keys cache refreshed: 31,570 keys

✅  Word Assembly Engine v11 ready


## CELL 4 - Intention Dataset + ML Classifier (TF-IDF + cosine)

In [6]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df_intent    = pd.read_csv(INTENTION_PATH)
INTENTION_MAP = {}
_intent_labels   = []
_intent_examples = []

for _, row in df_intent.iterrows():
    intent_en = str(row['intention_en']).strip()
    intent_ar = str(row['intention_ar']).strip() if 'intention_ar' in df_intent.columns else ''
    keywords  = [kw.strip().lower() for kw in str(row['keywords']).split(',')]
    example_text = ' '.join(keywords)
    if 'example_en' in df_intent.columns and pd.notna(row.get('example_en')):
        example_text += ' ' + str(row['example_en'])
    INTENTION_MAP[intent_en] = {'arabic': intent_ar, 'keywords': set(keywords)}
    _intent_labels.append(intent_en)
    _intent_examples.append(example_text)

_tfidf_vec     = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
_intent_matrix = _tfidf_vec.fit_transform(_intent_examples)

def get_intention_ml(text: str) -> dict:
    if not text or not text.strip():
        return {'intention_en': 'unknown', 'intention_ar': '', 'confidence': 0.0}
    query_vec = _tfidf_vec.transform([text.lower()])
    sims      = cosine_similarity(query_vec, _intent_matrix).flatten()
    best_idx  = int(np.argmax(sims))
    best_sim  = float(sims[best_idx])
    if best_sim >= 0.05:
        intent_en = _intent_labels[best_idx]
    else:
        tl, best, best_n = text.lower(), 'unknown', 0
        for intent, data in INTENTION_MAP.items():
            n = sum(1 for kw in data['keywords'] if kw.strip() in tl)
            if n > best_n:
                best_n, best = n, intent
        intent_en = best
    return {'intention_en': intent_en,
            'intention_ar': INTENTION_MAP.get(intent_en, {}).get('arabic', ''),
            'confidence'  : round(best_sim, 4)}

get_intention = get_intention_ml
print(f'✅ Intention ML classifier ready: {len(INTENTION_MAP)} classes')


✅ Intention ML classifier ready: 139 classes


## CELL 5 - TLA Deep Normalization v14 (confirmed ḥ→h mapping)

In [7]:
import re, unicodedata

EGYPTIAN_CHAR_MAP = {
    '\u1eec3': 'a',
    '\ua723': 'a',     # ꜣ Egyptian aleph
    '\ua725': 'a',     # ꜥ Egyptian ayin
    '\ua7bd': 'i',
    'y': 'y',
    '\u1eea5': 'a',
    'w': 'w', 'b': 'b', 'p': 'p', 'f': 'f',
    'm': 'm', 'n': 'n', 'r': 'r', 'h': 'h',
    '\u1e25': 'h',     # ḥ → h  ✅
    '\u1e2b': 'kh',    # ḫ → kh
    '\u1e96': 'kh',    # ẖ → kh
    's': 's',
    '\u0161': 'sh',    # š → sh
    '\u1e33': 'q',     # ḳ → q
    'q': 'q', 'k': 'k', 'g': 'g', 't': 't',
    '\u1e6f': 'tj',    # ṯ → tj
    'd': 'd',
    '\u1e0f': 'dj',    # ḏ → dj
    '\u1e6d': 't',     # ṭ → t
    '\u1e0d': 'd',     # ḍ → d
    '\u1e63': 's',     # ṣ → s
    '\u1e93': 'z',     # ẓ → z
    'i': 'i',
}

SUFFIXES_TO_REMOVE = ['=f','=k','=\u1e6f','=s','=sn','=\ua7bd','=n','=tn','=f\ua7bd']

def normalize_tla(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''
    text = re.sub(r'[()]', '', text)
    text = unicodedata.normalize('NFC', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    text = text.lower()
    for egy, lat in EGYPTIAN_CHAR_MAP.items():
        text = text.replace(egy.lower(), lat)
    for suf in SUFFIXES_TO_REMOVE:
        pattern = re.escape(suf) + r'(?=[\s\.]|$)'
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    text = re.sub(r'\.+', '.', text)
    text = text.replace('-', ' ')
    text = re.sub(r'\s+', ' ', text)
    text = text.strip('. ')
    return text

def normalize_for_display(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'[()]', '', text)
    for suf in SUFFIXES_TO_REMOVE:
        pattern = re.escape(suf) + r'(?=[\s\.]|$)'
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text)
    return text.strip('. ')

_norm_tests = [
    ('si\ua723t n si\ua725t \u1e25r', 'siat n siat hr'),
    ('n\u1e0f (w)di\u0307 r =s',       'ndj wdi r'),
]
print('Normalization tests:')
for raw, expected in _norm_tests:
    result = normalize_tla(raw)
    ok = '✅ OK' if result == expected else f'❌ GOT: {result} | EXPECTED: {expected}'
    print(f'  "{raw}" -> "{result}" [{ok}]')
print('✅ TLA normalization ready')


Normalization tests:
  "siꜣt n siꜥt ḥr" -> "siat n siat hr" [✅ OK]
  "nḏ (w)di̇ r =s" -> "ndj wdi r" [✅ OK]
✅ TLA normalization ready


## CELL 6 - Build FAISS + BM25 Indexes + Train/Test Split

In [8]:
import pickle, faiss, gc
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from rank_bm25 import BM25Okapi

EMBED_MODEL    = None
faiss_index    = None
bbaw_meta      = None
bm25_index     = None
df_test_global = None

if (os.path.exists(FAISS_INDEX_PATH) and
        os.path.exists(FAISS_META_PATH) and
        os.path.exists(BM25_CACHE_PATH)):
    print('Loading cached FAISS + BM25 indexes...')
    faiss_index = faiss.read_index(FAISS_INDEX_PATH)
    with open(FAISS_META_PATH, 'rb') as f:
        bbaw_meta = pickle.load(f)
    with open(BM25_CACHE_PATH, 'rb') as f:
        bm25_index = pickle.load(f)
    print(f'  FAISS vectors : {faiss_index.ntotal}')
    if os.path.exists(TEST_SET_PATH):
        df_test_global = pd.read_csv(TEST_SET_PATH)
        print(f'  Test set rows : {len(df_test_global)}')
    print('✅ Cached indexes loaded')
else:
    print('Building indexes from scratch...')
    dataset = load_dataset(
        'thesaurus-linguae-aegyptiae/tla-Earlier_Egyptian_original-v18-premium',
        split='train'
    )
    df_raw  = pd.DataFrame(dataset)
    cols_drop = [c for c in ['hieroglyphs','dateNotBefore','dateNotAfter'] if c in df_raw.columns]
    df_clean  = df_raw.drop(columns=cols_drop)
    df_clean  = df_clean.dropna(subset=['transliteration','translation'])
    df_clean  = df_clean[df_clean['transliteration'].str.strip() != '']
    df_clean  = df_clean[df_clean['translation'].str.strip() != '']
    df_clean  = df_clean.drop_duplicates(subset=['transliteration','translation'])
    df_clean  = df_clean.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f'  Clean records: {len(df_clean)}')
    split_idx      = int(len(df_clean) * TRAIN_SPLIT)
    df_train       = df_clean.iloc[:split_idx].copy()
    df_test_global = df_clean.iloc[split_idx:].copy()
    df_test_global.to_csv(TEST_SET_PATH, index=False)
    print(f'  Train: {len(df_train)} | Test: {len(df_test_global)}')
    EMBED_MODEL = SentenceTransformer('BAAI/bge-m3', device='cpu')
    norms       = df_train['transliteration'].apply(normalize_tla).tolist()
    print('  Encoding embeddings...')
    vectors     = EMBED_MODEL.encode(norms, batch_size=128, show_progress_bar=True,
                                      normalize_embeddings=True).astype('float32')
    dim         = vectors.shape[1]
    faiss_index = faiss.IndexFlatIP(dim)
    faiss_index.add(vectors)
    faiss.write_index(faiss_index, FAISS_INDEX_PATH)
    bbaw_meta = {'transcriptions': df_train['transliteration'].tolist(),
                  'translations'  : df_train['translation'].tolist()}
    with open(FAISS_META_PATH, 'wb') as f: pickle.dump(bbaw_meta, f)
    bm25_tokens = [normalize_tla(t).split() for t in df_train['transliteration']]
    bm25_index  = BM25Okapi(bm25_tokens)
    with open(BM25_CACHE_PATH, 'wb') as f: pickle.dump(bm25_index, f)
    print(f'  FAISS vectors : {faiss_index.ntotal}')
    print('✅ Indexes built and cached')


2026-04-09 02:13:52.397001: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775700832.617348     171 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775700832.682999     171 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775700833.209803     171 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775700833.209863     171 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775700833.209865     171 computation_placer.cc:177] computation placer alr

Building indexes from scratch...


README.md: 0.00B [00:00, ?B/s]

train.jsonl:   0%|          | 0.00/5.92M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

  Encoding embeddings...


Batches:   0%|          | 0/74 [00:00<?, ?it/s]

  FAISS vectors : 9408
✅ Indexes built and cached


## CELL 7 - Load Seed-X + Sentiment + spaCy

In [9]:
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast, pipeline
from safetensors import safe_open
import transformers.modeling_utils as _mu
import torch, gc, re, spacy

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

if 'seed_x_model' in globals():
    del globals()['seed_x_model']
gc.collect()
torch.cuda.empty_cache()

_orig_safe_open = safe_open
class _PatchedSafeOpen:
    def __init__(self, path, framework, device='cpu'):
        self._f = _orig_safe_open(path, framework=framework, device=device)
    def metadata(self):
        m = self._f.metadata()
        return m if (m is not None and m.get('format')) else {'format': 'pt'}
    def keys(self):          return self._f.keys()
    def get_tensor(self, k): return self._f.get_tensor(k)
    def __enter__(self):     return self
    def __exit__(self, *a):  pass
import safetensors._safetensors_rust as _sr
_sr.safe_open  = _PatchedSafeOpen
_mu.safe_open  = _PatchedSafeOpen

MODEL_PATH = 'ByteDance-Seed/Seed-X-PPO-7B' if ENV == 'kaggle' else 'seed-x-ppo-7b'

seed_x_tokenizer = PreTrainedTokenizerFast.from_pretrained(
    MODEL_PATH, trust_remote_code=True)
if seed_x_tokenizer.pad_token is None:
    seed_x_tokenizer.pad_token = seed_x_tokenizer.eos_token

seed_x_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    device_map='auto', trust_remote_code=True, low_cpu_mem_usage=True,
)
seed_x_model.eval()

LANG_TAGS = {'English': 'en', 'German': 'de', 'Arabic': 'ar',
             'French': 'fr', 'Italian': 'it', 'Spanish': 'es'}

sentiment_pipe = pipeline('sentiment-analysis',
                           model='cardiffnlp/twitter-roberta-base-sentiment-latest',
                           device=-1, top_k=None)

nlp_spacy = spacy.load('en_core_web_sm')
print('✅ Models loaded')


Device: cuda


tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'LlamaTokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/15.0G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✅ Models loaded


## CELL 8 - Core Helper Functions V10

In [10]:
from rapidfuzz import fuzz
import time

STANDALONE_PARTICLES = {'n', 'm', 'r', 'in', 'iw', 'nn', 'hr', '\u1e25r'}

def _get_phonetic(code: str, use_tla: bool = True) -> tuple:
    key  = code.strip().lower()
    info = GARDINER_MAP.get(key, {})
    if info.get('is_determinative', False):
        return '', True
    if use_tla:
        ph = info.get('phonetic_tla', '') or info.get('phonetic', '')
    else:
        ph = info.get('phonetic', '') or info.get('phonetic_tla', '')
    return (ph if ph else key), False

def assemble_words(codes: list, use_tla: bool = True) -> list:
    words        = []
    current_word = []
    prev_was_det = True
    for c in codes:
        phonetic, is_det = _get_phonetic(c, use_tla=use_tla)
        if is_det:
            if current_word:
                words.append(''.join(current_word))
                current_word = []
            prev_was_det = True
        else:
            if phonetic:
                if prev_was_det and phonetic.lower() in STANDALONE_PARTICLES and not current_word:
                    words.append(phonetic)
                    prev_was_det = False
                else:
                    current_word.append(phonetic)
                    prev_was_det = False
    if current_word:
        words.append(''.join(current_word))
    return [w for w in words if w]

def assemble_tla(codes: list) -> str:
    return ' '.join(assemble_words(codes, use_tla=True))

def assemble_standard(codes: list) -> str:
    return ' '.join(assemble_words(codes, use_tla=False))

def assemble_raw_with_det(codes: list) -> str:
    parts, current = [], []
    for c in codes:
        ph, is_det = _get_phonetic(c, use_tla=True)
        if is_det:
            if current: parts.append(''.join(current)); current = []
            parts.append(f'[{c.upper()}]')
        elif ph:
            current.append(ph)
    if current: parts.append(''.join(current))
    return ' '.join(parts)

def get_meanings(codes: list) -> list:
    meanings = []
    for c in codes:
        key  = c.strip().lower()
        info = GARDINER_MAP.get(key, {})
        if not info.get('is_determinative', False):
            m = info.get('meaning', '')
            if m: meanings.append(m)
    return meanings

def get_embed_model():
    global EMBED_MODEL
    if EMBED_MODEL is None:
        EMBED_MODEL = SentenceTransformer('BAAI/bge-m3', device='cpu')
    return EMBED_MODEL

def fix_grammar(text: str) -> str:
    doc   = nlp_spacy(text)
    sents = [s.text.strip() for s in doc.sents]
    return ' '.join(s[0].upper() + s[1:] for s in sents if s)

def get_sentiment(text: str) -> dict:
    raw     = sentiment_pipe(text[:512])[0][0]
    label   = raw['label'].lower()
    mapping = {'positive': 'Positive', 'neutral': 'Neutral', 'negative': 'Negative'}
    return {'label': mapping.get(label, label), 'confidence': raw['score']}

def translate_seedx(text: str, src_lang: str, tgt_lang: str,
                     max_new_tokens: int = 150) -> str:
    """Generic translation via Seed-X. Used only for Arabic translation."""
    tgt_tag = LANG_TAGS.get(tgt_lang, 'en')
    prompt  = f'Translate the following text into {tgt_lang}:\n{text}\n<{tgt_tag}>'
    inputs  = seed_x_tokenizer(prompt, return_tensors='pt',
                                truncation=True, max_length=512)
    inputs.pop('token_type_ids', None)
    dev    = next(seed_x_model.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    with torch.no_grad():
        out = seed_x_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            num_beams=2, do_sample=False,
            no_repeat_ngram_size=3, repetition_penalty=1.4,
            pad_token_id=seed_x_tokenizer.pad_token_id,
            eos_token_id=seed_x_tokenizer.eos_token_id,
        )
    new_tok = out[0][inputs['input_ids'].shape[1]:]
    decoded = seed_x_tokenizer.decode(new_tok, skip_special_tokens=True).strip()
    parts   = re.split(r'[\n]|(?<=[.!?])\s', decoded)
    clean   = parts[0].strip() if parts else decoded
    return re.sub(r'[^\w\s\u0600-\u06FF.,!?\'\"-]', '', clean).strip()

def translate_arabic_verified(english_text: str, max_retries: int = 2) -> dict:
    arabic  = translate_seedx(english_text, 'English', 'Arabic', max_new_tokens=200)
    back_en = translate_seedx(arabic, 'Arabic', 'English', max_new_tokens=150)
    orig_words = set(english_text.lower().split())
    back_words = set(back_en.lower().split())
    overlap    = len(orig_words & back_words) / max(len(orig_words), 1)
    return {'arabic': arabic, 'verified': overlap >= 0.25,
            'back_translation': back_en, 'confidence': round(overlap, 3)}

def interpret_rag_score(score: float, k: int = RRF_K, alpha: float = HYBRID_ALPHA) -> str:
    max_score = 1.0 / (k + 1)
    ratio     = score / max_score if max_score > 0 else 0
    if ratio >= 0.90: return f'✅ Excellent  ({score:.4f} = {ratio:.0%} of max {max_score:.4f})'
    elif ratio >= 0.60: return f'👍 Good       ({score:.4f} = {ratio:.0%} of max {max_score:.4f})'
    elif ratio >= 0.30: return f'⚠️  Weak       ({score:.4f} = {ratio:.0%} of max {max_score:.4f})'
    else: return f'❌ No match   ({score:.4f} = {ratio:.0%} of max {max_score:.4f})'

print('✅ Core helpers ready (V10 + v14 refinements)')


✅ Core helpers ready (V10 + v14 refinements)


## CELL 9 - Hybrid Search (Dense FAISS + BM25 + RRF)

In [11]:
def _dense_search(query_norm: str, top_k: int) -> list:
    model = get_embed_model()
    q_vec = model.encode([query_norm], normalize_embeddings=True).astype('float32')
    if hasattr(faiss_index, 'nprobe'): faiss_index.nprobe = 10
    D, I = faiss_index.search(q_vec, top_k * 2)
    results = []
    for score, idx in zip(D[0], I[0]):
        if idx < 0: continue
        results.append({'idx': int(idx), 'score': float(score),
                         'transcription': bbaw_meta['transcriptions'][idx],
                         'translation'  : bbaw_meta['translations'][idx]})
    return results

def _bm25_search(query_norm: str, top_k: int) -> list:
    tokens      = query_norm.split()
    bm25_scores = bm25_index.get_scores(tokens)
    top_indices = np.argsort(bm25_scores)[::-1][:top_k * 2]
    results = []
    for idx in top_indices:
        if bm25_scores[idx] > 0:
            results.append({'idx': int(idx), 'score': float(bm25_scores[idx]),
                             'transcription': bbaw_meta['transcriptions'][idx],
                             'translation'  : bbaw_meta['translations'][idx]})
    return results

def _rrf(dense: list, sparse: list,
          k: int = RRF_K, alpha: float = HYBRID_ALPHA) -> list:
    scores = {}
    for rank, r in enumerate(dense):
        scores[r['idx']] = scores.get(r['idx'], 0) + alpha * (1 / (k + rank + 1))
    for rank, r in enumerate(sparse):
        scores[r['idx']] = scores.get(r['idx'], 0) + (1 - alpha) * (1 / (k + rank + 1))
    all_hits = {r['idx']: r for r in dense + sparse}
    return [{'rrf_score': sc, 'score': sc, **all_hits[idx]}
            for idx, sc in sorted(scores.items(), key=lambda x: -x[1])]

# FIX C: Multi-Query helpers
def _build_search_queries(tla_tokens: list, codes: list) -> list:
    """
    Builds multiple query variants to maximize RAG recall for compound words.
    Example for S42 D2 F34 → ['ḫrp','ḥr','ꞽb']:
      Q1 (joined)     : 'khrphrib'    — compound form
      Q2 (spaced)     : 'khrp hr ib'  — morphemic form
      Q3 (hyphenated) : 'khrp hr ib'  — normalize_tla converts - to space (same as Q2)
      Q4 (Gardiner)   : 'khrp hr ib'  — from GARDINER_MAP phonetics
    The key insight: Q2/Q3/Q4 allow BM25 to match individual morphemes like 'ḫrp' in 'ḫrp-ḥr-ꞽb'
    """
    queries = set()

    # Q1: compound (joined, no separator)
    q1 = normalize_tla(''.join(tla_tokens))
    if q1: queries.add(q1)

    # Q2: space-separated morphemes
    q2 = normalize_tla(' '.join(tla_tokens))
    if q2: queries.add(q2)

    # Q3: from GARDINER_MAP — use the raw phonetics (most faithful to TLA dataset)
    if codes:
        phonetics = []
        for c in codes:
            info = GARDINER_MAP.get(c.strip().lower(), {})
            ph = info.get('phonetic_tla', '') or info.get('phonetic', '')
            if ph and not info.get('is_determinative', False):
                phonetics.append(ph)
        if phonetics:
            q3 = normalize_tla(' '.join(phonetics))
            if q3: queries.add(q3)
            # Also try with hyphen to match 'ḫrp-ḥr(.ꞽ)-ꞽb' style
            q3h = normalize_tla('-'.join(phonetics))
            if q3h: queries.add(q3h)

    return [q for q in queries if q and q.strip()]


def hybrid_search(query_raw: str, top_k: int = TOP_K_RAG,
                   tla_tokens: list = None, codes: list = None) -> list:
    """
    FIX C: Multi-Query Hybrid Search.
    Sends multiple query variants and fuses results with RRF.
    Fixes the S42+D2+F34 → 'ḫrp-ḥr-ꞽb' → 'Leiter der Mitte' retrieval.
    """
    if tla_tokens:
        queries = _build_search_queries(tla_tokens, codes or [])
    else:
        queries = [normalize_tla(query_raw)]

    if not queries:
        queries = [normalize_tla(query_raw)]

    # Run search for each query variant
    dense_all  = []
    sparse_all = []
    for q in queries:
        dense_all.extend(_dense_search(q, top_k))
        sparse_all.extend(_bm25_search(q, top_k))

    # Deduplicate: keep best score per idx
    dense_best  = {}
    for r in dense_all:
        if r['idx'] not in dense_best or r['score'] > dense_best[r['idx']]['score']:
            dense_best[r['idx']] = r
    sparse_best = {}
    for r in sparse_all:
        if r['idx'] not in sparse_best or r['score'] > sparse_best[r['idx']]['score']:
            sparse_best[r['idx']] = r

    merged = _rrf(list(dense_best.values()), list(sparse_best.values()))
    return merged[:top_k]

print('✅ Multi-Query Hybrid Search v15 ready')
print(f'   k={RRF_K}, alpha={HYBRID_ALPHA}')
print(f'   Max RRF score = {RRF_SCORE_MAX:.4f}')


✅ Hybrid search ready
   Using k=60, alpha=0.6
   Theoretical max RRF score = 0.0164


## CELL 10 - RAG Prompt + Translation v14
### FIX 2 + 3: Extract German from RAG directly → strict EN translation prompt

In [12]:
import re as _re

# ── v14: Extended garbage patterns ──────────────────────────────
_LLM_GARBAGE_PATTERNS = [
    r'^Output\b',
    r'^These are (translations|examples)',
    r'^Translation[:\s]',
    r'^German\s*[Tt]ranslation[:\s]',
    r'^Translate\b',
    r'^you are (a|an|the)\b',
    r'^i am (a|an|the)\b',
    r'\<[a-z]{2}\>$',
    r'^OPINION\b',           # v14: was leaking meta-commentary
    r'^COT\b',               # v14: chain-of-thought prefix
    r'^THOUGHTS\b',          # v14: explicit hallucination pattern
    r'^This is a translation', # v14
    r'hieroglyphic',           # v14: meta-commentary not translation
    r'^An in-depth analysis',  # v14
]

_GARBAGE_RE = _re.compile(
    '|'.join(_LLM_GARBAGE_PATTERNS),
    _re.IGNORECASE
)

def _clean_llm_output(raw_output: str) -> str:
    """Remove COT / prompt echo / garbage lines from Seed-X output."""
    if not raw_output:
        return ''
    lines = [ln.strip() for ln in raw_output.split('\n')]
    clean_lines = []
    for ln in lines:
        if not ln: continue
        ln = _re.sub(r'\s*<[a-z]{2}>\s*$', '', ln).strip()
        if not ln: continue
        if _GARBAGE_RE.search(ln): continue
        clean_lines.append(ln)
    return clean_lines[0] if clean_lines else raw_output.strip()

def _count_tokens(text: str) -> int:
    return len(text) // 4

def _get_best_german_from_rag(rag_results: list) -> str:
    """
    v14 FIX 2: Extract German translation directly from the top RAG hit.
    This is the key fix — instead of asking Seed-X to generate a German
    translation (which causes hallucinations), we use the actual German
    translation that already exists in the dataset corpus.
    """
    for r in rag_results:
        translation = r.get('translation', '').strip()
        if len(translation.replace('.', '').replace(' ', '')) > 10:
            return translation
    return ''

def german_to_english_strict(german_text: str,
                               max_new_tokens: int = 150) -> str:
    """
    v14 FIX 3: 2-line strict prompt — no role, no context, no examples.
    The old translate_seedx() with a generic prompt caused hallucinations
    like 'THOUGHTS' or 'OPINION An in-depth analysis...'
    This prompt forces the model to output ONLY the English translation.
    Falls back gracefully to the German text if Seed-X still hallucinates.
    """
    prompt = (
        f'Translate from German to English. Output only the translation.\n'
        f'German: {german_text}\n'
        f'English:'
    )
    inputs = seed_x_tokenizer(prompt, return_tensors='pt',
                               truncation=True, max_length=256)
    inputs.pop('token_type_ids', None)
    dev    = next(seed_x_model.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    with torch.no_grad():
        out = seed_x_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            num_beams=2, do_sample=False,
            no_repeat_ngram_size=3, repetition_penalty=1.4,
            pad_token_id=seed_x_tokenizer.pad_token_id,
            eos_token_id=seed_x_tokenizer.eos_token_id,
        )
    new_tok = out[0][inputs['input_ids'].shape[1]:]
    raw     = seed_x_tokenizer.decode(new_tok, skip_special_tokens=True).strip()
    cleaned = _clean_llm_output(raw)
    if not cleaned or set(cleaned) <= set('. -'):
        return german_text   # graceful fallback: show German if translation fails
    return cleaned

def translate_with_rag_prompt(query_original, query_norm, retrieved, meanings,
                               assembled_words=None) -> str:
    """
    v14: Returns the best German text for this query.
    Priority:
      1. Top RAG hit translation (from corpus, reliable)
      2. LLM-generated German using examples (fallback, may hallucinate)
    The caller (sentence_path) then calls german_to_english_strict().
    """
    if assembled_words is None:
        assembled_words = query_norm.split()

    # Step 1: Try to get German directly from RAG corpus
    best_german = _get_best_german_from_rag(retrieved)
    if best_german:
        return best_german

    # Step 2: No good RAG result — generate via LLM with examples
    meaning_hint  = ', '.join(m for m in meanings if m)
    words_display = ' '.join(assembled_words) if assembled_words else query_original

    header = (
        'You are an expert in Earlier Egyptian. '
        'Translate the transliteration into German. '
        'Output ONLY the German translation. No explanations.\n\n'
        f'Transliteration: {query_original}\n'
        f'Words: {words_display}\n'
        f'Sign meanings: {meaning_hint or "N/A"}\n\n'
        'Closest corpus examples:\n'
    )
    footer = '\nGerman Translation:'

    budget = _count_tokens(header) + _count_tokens(footer)
    examples_text = ''
    included = 0
    for i, ex in enumerate(retrieved, 1):
        block = (f'\n{i}. [{ex["transcription"]}] -> {ex["translation"]}')
        if budget + _count_tokens(block) > MAX_PROMPT_TOKENS:
            break
        examples_text += block
        budget += _count_tokens(block)
        included += 1

    print(f'  [Prompt fallback] {included}/{len(retrieved)} examples | ~{budget} tokens')
    prompt = header + examples_text + footer
    inputs = seed_x_tokenizer(prompt, return_tensors='pt',
                               truncation=True, max_length=4096)
    inputs.pop('token_type_ids', None)
    dev    = next(seed_x_model.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    with torch.no_grad():
        out = seed_x_model.generate(
            **inputs, max_new_tokens=200, num_beams=2, do_sample=False,
            no_repeat_ngram_size=3, repetition_penalty=1.4,
            pad_token_id=seed_x_tokenizer.pad_token_id,
            eos_token_id=seed_x_tokenizer.eos_token_id,
        )
    new_tok = out[0][inputs['input_ids'].shape[1]:]
    raw     = seed_x_tokenizer.decode(new_tok, skip_special_tokens=True).strip()
    cleaned = _clean_llm_output(raw)
    if not cleaned:
        return meaning_hint or query_original
    return cleaned

print('✅ Token-aware RAG prompt builder ready (v14)')


✅ Token-aware RAG prompt builder ready (v14)


## CELL 10.5 — Translation Cache (FIX F: disk-persistent)
### Speeds up repeated German→English translations by ~35s per call

In [ ]:
import hashlib
import pickle

# FIX F: Persistent translation cache
_TRANSLATION_CACHE      : dict[str, str] = {}
_TRANSLATION_CACHE_PATH = os.path.join(_WORK_ROOT, 'translation_cache.pkl')

def _load_translation_cache():
    global _TRANSLATION_CACHE
    if os.path.exists(_TRANSLATION_CACHE_PATH):
        try:
            with open(_TRANSLATION_CACHE_PATH, 'rb') as f:
                _TRANSLATION_CACHE = pickle.load(f)
            print(f'✅ Translation cache loaded: {len(_TRANSLATION_CACHE)} entries')
        except Exception:
            _TRANSLATION_CACHE = {}
            print('⚠️  Translation cache corrupted — starting fresh')
    else:
        print('   Translation cache: empty (first run)')

def _save_translation_cache():
    try:
        with open(_TRANSLATION_CACHE_PATH, 'wb') as f:
            pickle.dump(_TRANSLATION_CACHE, f)
    except Exception as e:
        print(f'⚠️  Failed to save translation cache: {e}')

_load_translation_cache()

_german_to_english_strict_original = german_to_english_strict

def german_to_english_strict(german_text: str, max_new_tokens: int = 150) -> str:
    """
    FIX F: Cached wrapper — checks in-memory + disk cache before calling Seed-X.
    Same interface, transparent to callers.
    """
    if not german_text or not german_text.strip():
        return ''
    cache_key = hashlib.md5(german_text.strip().encode('utf-8')).hexdigest()
    if cache_key in _TRANSLATION_CACHE:
        return _TRANSLATION_CACHE[cache_key]
    result = _german_to_english_strict_original(german_text, max_new_tokens=max_new_tokens)
    _TRANSLATION_CACHE[cache_key] = result
    _save_translation_cache()
    return result

print('✅ Translation cache layer active')
print(f'   Cache size: {len(_TRANSLATION_CACHE)} entries')
print(f'   Cache file: {_TRANSLATION_CACHE_PATH}')


## CELL 11 - Path Handlers v14
### FIX 1: WORD PATH removed — sentence_path() is the only active path

In [13]:
def sentence_path(codes, rag_results, meanings):
    """
    v14: All RAG-successful inputs go through here.
    1. Get German from RAG corpus (or LLM fallback)
    2. Translate German → English with strict 2-line prompt
    3. Apply grammar fix, sentiment, intention, Arabic
    """
    assembled_words = assemble_words(codes, use_tla=True)
    query_orig      = ' '.join(assembled_words)
    query_norm      = normalize_tla(query_orig)

    # Step 1: German from RAG or LLM
    german = translate_with_rag_prompt(
        query_orig, query_norm, rag_results, meanings,
        assembled_words=assembled_words
    )

    # Guard empty/garbage German
    if not german or set(german.strip()) <= set('. -') or len(german.strip()) < 3:
        meaning_hint = ', '.join(m for m in meanings if m)
        german = meaning_hint or query_orig

    # Step 2: German → English (strict prompt)
    english = fix_grammar(german_to_english_strict(german))

    if not english or set(english.strip()) <= set('. -'):
        meaning_hint = ', '.join(m for m in meanings if m)
        english = meaning_hint if meaning_hint else query_orig

    sentiment = get_sentiment(english)
    intention = get_intention(english)
    ar_result = translate_arabic_verified(english)
    best      = rag_results[0]
    return {
        'path'             : 'sentence',
        'english'          : english,
        'arabic'           : ar_result['arabic'],
        'arabic_verified'  : ar_result['verified'],
        'arabic_confidence': ar_result['confidence'],
        'sentiment'        : sentiment,
        'intention'        : intention,
        'rag_score'        : best['score'],
        'rag_source'       : best['transcription'],
        'rag_results'      : rag_results,
        'assembled_words'  : assembled_words,
        'german_source'    : german,   # v14: expose for debug
    }

print('✅ Path handlers ready (v14 — WORD PATH removed)')


✅ Path handlers ready (v14 — WORD PATH removed)


## CELL 12 - Full Pipeline v14
### FIX 5: 2 paths only — SENTENCE (RAG ≥ threshold) or FALLBACK

In [14]:
def full_pipeline(gardiner_input: str, verbose: bool = True) -> dict:
    t0              = time.time()
    codes           = gardiner_input.strip().upper().split()
    assembled_words = assemble_words(codes, use_tla=True)
    tla_query       = ' '.join(assembled_words)
    tla_norm        = normalize_tla(tla_query)
    meanings        = get_meanings(codes)

    if verbose:
        print(f'Input codes   : {codes}')
        print(f'Raw + det     : {assemble_raw_with_det(codes)}')
        print(f'Assembled     : {assembled_words}')
        print(f'TLA query     : {tla_query}')
        print(f'TLA norm      : {tla_norm}')

    wa = word_assembly_result(assembled_words)

    if verbose:
        print(f'Word Assembly     : {wa["best_segmentation"]}')
        print(f'Assembly Score    : {wa["best_score"]}')
        print(f'Assembly types    : {[m["type"] for m in wa["best_metadata"]]}')
        if wa["alternatives"]:
            print(f'Alternatives      : {wa["alternatives"][:2]}')
        for seg_word in wa['best_segmentation']:
            vocab_entry = WORD_VOCAB.get(seg_word.lower(), {})
            if vocab_entry:
                print(f'  [{seg_word}] → source={vocab_entry.get("source","")}')

    # FIX C: Multi-query hybrid search — passes tla_tokens + codes
    rag_results = hybrid_search(
        tla_query,
        top_k   = TOP_K_RAG,
        tla_tokens = assembled_words,
        codes      = codes,
    )
    best_score  = rag_results[0]['score'] if rag_results else 0.0

    if verbose:
        print(f'Hybrid RAG best RRF score: {best_score:.4f}')
        print(f'RAG Score Quality : {interpret_rag_score(best_score)}')
        print(f'SIMILARITY_THRESHOLD = {SIMILARITY_THRESHOLD:.4f}')
        if rag_results:
            top = rag_results[0]
            print(f'Top RAG hit   : [{top["transcription"]}] → {top["translation"]}')

    if best_score >= SIMILARITY_THRESHOLD:
        if verbose: print('Path: SENTENCE  ← RAG retrieval succeeded')
        result = sentence_path(codes, rag_results, meanings)
    else:
        if verbose: print('Path: FALLBACK  ← RAG score below threshold, using LLM')
        per_word_matches = [m['meaning'] for m in wa['best_metadata'] if m.get('meaning')]
        word_meanings    = [m for m in meanings if m]
        meaning_hint     = ', '.join(word_meanings)
        if per_word_matches:
            ctx     = ' '.join(per_word_matches)
            english = fix_grammar(german_to_english_strict(ctx))
        else:
            best_german = _get_best_german_from_rag(rag_results)
            if best_german:
                english = fix_grammar(german_to_english_strict(best_german))
            elif word_meanings:
                ctx = f'Ancient Egyptian signs meaning: {meaning_hint}.'
                english = fix_grammar(german_to_english_strict(ctx))
            else:
                english = tla_query
        if not english or set(english.strip()) <= set('. -'):
            english = meaning_hint if word_meanings else tla_query
        ar_result = translate_arabic_verified(english)
        result = {
            'path'             : 'fallback',
            'english'          : english,
            'arabic'           : ar_result['arabic'],
            'arabic_verified'  : ar_result['verified'],
            'arabic_confidence': ar_result['confidence'],
            'assembled_words'  : assembled_words,
            'sentiment'        : get_sentiment(english),
            'intention'        : get_intention(english),
            'rag_score'        : best_score,
            'rag_source'       : rag_results[0]['transcription'] if rag_results else '',
            'rag_results'      : rag_results,
        }

    result.update({
        'input'            : gardiner_input,
        'tla_phonemes'     : tla_query,
        'tla_normalised'   : tla_norm,
        'meanings'         : meanings,
        'assembly_metadata': wa['best_metadata'],
        'assembled_words'  : result.get('assembled_words', assembled_words),
        'assembly_alts'    : wa['alternatives'],
        'assembly_score'   : wa['best_score'],
        'processing_time'  : round(time.time() - t0, 2),
    })

    if verbose:
        print('=' * 60)
        print(f'PATH          : {result["path"].upper()}')
        print(f'Assembled     : {" ".join(result.get("assembled_words", []))}')
        print(f'TLA norm      : {tla_norm}')
        if 'german_source' in result:
            print(f'German (RAG)  : {result["german_source"]}')
        print(f'English       : {result["english"]}')
        print(f'Arabic        : {result["arabic"]}')
        ar_ver = result.get('arabic_verified')
        if ar_ver is not None:
            lbl = 'verified' if ar_ver else 'low confidence'
            print(f'Arabic check  : {lbl} ({result.get("arabic_confidence", 0):.0%})')
        s = result.get('sentiment')
        if s: print(f'Sentiment     : {s["label"]} ({s["confidence"]:.0%})')
        i = result.get('intention')
        if i: print(f'Intention     : {i["intention_en"]} / {i["intention_ar"]}')
        rag_s = result['rag_score']
        print(f'RAG Score     : {rag_s:.4f}  →  {interpret_rag_score(rag_s)}')
        print(f'Time          : {result["processing_time"]}s')
        print('=' * 60)
    return result


# ── SMOKE TEST v15 ─────────────────────────────────────────────────────────────
print('\n=== SMOKE TEST v15 ===')
print('Test 1: S42 D2 F34  →  expect: Leiter der Mitte')
result1 = full_pipeline('S42 D2 F34')
print()
print('Test 2 (regression): S29 M17 G1 V13 D57 N35 S29 M17 D36 V13 D57 G5')
print('Expect: Verletzung für den, der Horus verletzt! → Injury for the one who injures Horus!')
result2 = full_pipeline('S29 M17 G1 V13 D57 N35 S29 M17 D36 V13 D57 G5')



=== SMOKE TEST v14 ===
Input  : S29 M17 G1 V13 D57 N35 S29 M17 D36 V13 D57 G5
Expect : German (RAG) = Verletzung für den, der Horus verletzt!
Expect : English      = Injury for the one who injures Horus!
Expect : Arabic       = إصابة لمن يؤذي حورس!
Expect : Path         = SENTENCE

Input codes   : ['S29', 'M17', 'G1', 'V13', 'D57', 'N35', 'S29', 'M17', 'D36', 'V13', 'D57', 'G5']
Raw + det     : siꜣt [D57] nsiꜥt [D57] ḥr
Assembled     : ['siꜣt', 'n', 'siꜥt', 'ḥr']
TLA query     : siꜣt n siꜥt ḥr
TLA norm      : siat n siat hr
Word Assembly     : ['siꜣt', 'n', 'siꜥt', 'ḥr']
Assembly Score    : 7.2
Assembly types    : ['fuzzy', 'exact', 'fuzzy', 'fuzzy']
Alternatives      : [['siꜣt', 'nsiꜥt', 'ḥr'], ['siꜣtn', 'siꜥt', 'ḥr']]
  [n] → vocab word_length=1, source=bbaw
Hybrid RAG best RRF score: 31.3685
RAG Score Quality : ✅ Excellent  (31.3685 = 191348% of max 0.0164)
SIMILARITY_THRESHOLD = 0.0082
Path: SENTENCE  ← RAG retrieval succeeded
PATH          : SENTENCE
Assembled     : siꜣt n siꜥt ḥ

## CELL 13 - Full Evaluation Suite (Translation + Retrieval)

In [15]:
import nltk
from nltk.translate.bleu_score   import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score                 import rouge_scorer
from sacrebleu.metrics           import CHRF
import warnings; warnings.filterwarnings('ignore')

nltk.download('wordnet', quiet=True)
nltk.download('punkt',   quiet=True)
nltk.download('omw-1.4', quiet=True)

def calc_bleu(reference, hypothesis):
    ref_tok = reference.lower().split()
    hyp_tok = hypothesis.lower().split()
    if not ref_tok or not hyp_tok: return 0.0
    smoother = SmoothingFunction().method1
    max_n    = min(len(ref_tok), len(hyp_tok), 4)
    weights  = {1:(1,0,0,0), 2:(.5,.5,0,0), 3:(.33,.33,.34,0)}.get(max_n, (.25,.25,.25,.25))
    return sentence_bleu([ref_tok], hyp_tok, weights=weights, smoothing_function=smoother) * 100

def calc_rouge(reference, hypothesis):
    scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    scores = scorer.score(reference.lower(), hypothesis.lower())
    return {k: v.fmeasure * 100 for k, v in scores.items()}

def calc_meteor(reference, hypothesis):
    try:
        return meteor_score([reference.lower().split()], hypothesis.lower().split()) * 100
    except: return 0.0

def calc_chrf(reference, hypothesis):
    return CHRF().sentence_score(hypothesis, [reference]).score

def calc_exact_match(reference, hypothesis):
    return 100.0 if reference.lower().strip() == hypothesis.lower().strip() else 0.0

def calc_word_overlap(reference, hypothesis):
    ref_words = set(reference.lower().split())
    hyp_words = set(hypothesis.lower().split())
    if not ref_words: return 0.0
    return len(ref_words & hyp_words) / len(ref_words) * 100

def calc_recall_at_k(reference_german, retrieved, k_values=[1,3,5,10,20]):
    ref_norm = normalize_tla(reference_german).lower()
    scores   = {}
    for k in k_values:
        hit = any(
            fuzz.ratio(normalize_tla(r['translation']).lower(), ref_norm) >= 70
            for r in retrieved[:k]
        )
        scores[f'recall@{k}'] = 100.0 if hit else 0.0
    return scores

def calc_mrr(reference_german, retrieved):
    ref_norm = normalize_tla(reference_german).lower()
    for rank, r in enumerate(retrieved, 1):
        if fuzz.ratio(normalize_tla(r['translation']).lower(), ref_norm) >= 70:
            return 100.0 / rank
    return 0.0

def evaluate_pipeline(gardiner_input, reference_english='', reference_german=''):
    result = full_pipeline(gardiner_input, verbose=False)
    hyp_en = result.get('english', '')
    rouge  = calc_rouge(reference_english, hyp_en) if reference_english else {}
    recall = calc_recall_at_k(reference_german, result.get('rag_results', [])) if reference_german else {}
    mrr    = calc_mrr(reference_german, result.get('rag_results', [])) if reference_german else 0.0
    avg_r  = np.mean(list(recall.values())) if recall else 0.0
    return {
        **result,
        'reference_english'  : reference_english,
        'bleu'               : calc_bleu(reference_english, hyp_en)        if reference_english else 0.0,
        'rouge1'             : rouge.get('rouge1', 0.0),
        'rouge2'             : rouge.get('rouge2', 0.0),
        'rougeL'             : rouge.get('rougeL', 0.0),
        'meteor'             : calc_meteor(reference_english, hyp_en)       if reference_english else 0.0,
        'chrf'               : calc_chrf(reference_english, hyp_en)         if reference_english else 0.0,
        'exact_match'        : calc_exact_match(reference_english, hyp_en)  if reference_english else 0.0,
        'word_overlap'       : calc_word_overlap(reference_english, hyp_en) if reference_english else 0.0,
        **recall,
        'mrr'                : mrr,
        'avg_retrieval_score': avg_r,
    }

print('✅ Evaluation suite ready')


✅ Evaluation suite ready


## CELL 14 - RAG vs LLM-Only Full Comparison

In [16]:
def translate_llm_only(query_original: str) -> str:
    """Direct Seed-X translation WITHOUT any RAG context."""
    prompt = (
        'You are an expert linguist specialising in Earlier Egyptian grammar.\n\n'
        'Translate the following Earlier Egyptian transliteration into German.\n'
        'Output ONLY the German translation.\n\n'
        f'Egyptian Transliteration:\n{query_original}\n\n'
        'German Translation:\n'
    )
    inputs = seed_x_tokenizer(prompt, return_tensors='pt',
                               truncation=True, max_length=512)
    inputs.pop('token_type_ids', None)
    dev    = next(seed_x_model.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    with torch.no_grad():
        out = seed_x_model.generate(
            **inputs, max_new_tokens=150, num_beams=1, do_sample=False,
            no_repeat_ngram_size=3, repetition_penalty=1.5,
            pad_token_id=seed_x_tokenizer.pad_token_id,
            eos_token_id=seed_x_tokenizer.eos_token_id,
        )
    new_tok = out[0][inputs['input_ids'].shape[1]:]
    return seed_x_tokenizer.decode(new_tok, skip_special_tokens=True).strip()


def run_full_comparison(num_samples: int = None):
    if df_test_global is None or len(df_test_global) == 0:
        print('❌ Test set not available. Run CELL 6 first.')
        return None, None
    test_df = df_test_global if num_samples is None else df_test_global.head(num_samples)
    print('=' * 70)
    print(f'🆚 RAG vs LLM-ONLY COMPARISON ({len(test_df)} samples)')
    print('=' * 70)
    rag_results_list = []
    llm_results_list = []
    for idx in range(len(test_df)):
        row           = test_df.iloc[idx]
        query_orig    = row['transliteration']
        reference_de  = row['translation']
        reference_en  = german_to_english_strict(reference_de)
        rag = evaluate_pipeline(query_orig, reference_english=reference_en,
                                reference_german=reference_de)
        rag_results_list.append(rag)
        llm_german  = translate_llm_only(query_orig)
        llm_english = german_to_english_strict(llm_german)
        rouge_llm   = calc_rouge(reference_en, llm_english)
        llm_results_list.append({
            'bleu'      : calc_bleu(reference_en, llm_english),
            'rouge1'    : rouge_llm.get('rouge1', 0.0),
            'rouge2'    : rouge_llm.get('rouge2', 0.0),
            'rougeL'    : rouge_llm.get('rougeL', 0.0),
            'meteor'    : calc_meteor(reference_en, llm_english),
            'chrf'      : calc_chrf(reference_en, llm_english),
            'exact_match'  : calc_exact_match(reference_en, llm_english),
            'word_overlap' : calc_word_overlap(reference_en, llm_english),
            'recall@1': 0.0, 'recall@3': 0.0, 'recall@5': 0.0,
            'recall@10': 0.0, 'recall@20': 0.0, 'mrr': 0.0,
        })
        if (idx + 1) % 10 == 0:
            print(f'  Processed {idx + 1}/{len(test_df)} samples...')
    df_rag = pd.DataFrame(rag_results_list)
    df_llm = pd.DataFrame(llm_results_list)
    metric_cols = ['bleu','rouge1','rouge2','rougeL','meteor','chrf',
                   'exact_match','word_overlap','recall@1','recall@3',
                   'recall@5','recall@10','recall@20','mrr']
    print('\n📊 COMPARISON SUMMARY')
    print(f'{"Metric":<20} {"RAG":>10} {"LLM-Only":>10} {"Δ":>10}')
    print('-' * 55)
    for m in metric_cols:
        rag_mean = df_rag[m].mean() if m in df_rag.columns else 0.0
        llm_mean = df_llm[m].mean() if m in df_llm.columns else 0.0
        delta    = rag_mean - llm_mean
        bar      = '▲' if delta > 0 else ('▼' if delta < 0 else '─')
        print(f'{m:<20} {rag_mean:>10.2f} {llm_mean:>10.2f} {bar}{abs(delta):>8.2f}')
    df_rag.to_csv(os.path.join(_WORK_ROOT, 'rag_eval_results.csv'), index=False)
    df_llm.to_csv(os.path.join(_WORK_ROOT, 'llm_eval_results.csv'), index=False)
    print('\n💾 Results saved to CSV files')
    return df_rag, df_llm

# df_rag_results, df_llm_results = run_full_comparison(num_samples=50)


## CELL 15 - K-Value Optimization

In [17]:
def run_k_optimization(num_samples: int = 10,
                        k_min: int = 5, k_max: int = 205, k_step: int = 5):
    global TOP_K_RAG
    if df_test_global is None or len(df_test_global) == 0:
        print('❌ Test set not available. Run CELL 6 first.')
        return None
    k_values = list(range(k_min, k_max, k_step))
    test_df  = df_test_global.head(num_samples)
    print('=' * 70)
    print(f'🔬 K-VALUE OPTIMIZATION ({num_samples} samples, K={k_min}..{k_max-1})')
    print('=' * 70)
    test_samples = []
    for idx in range(len(test_df)):
        q_orig = test_df.iloc[idx]['transliteration']
        ref_de = test_df.iloc[idx]['translation']
        ref_en = german_to_english_strict(ref_de)
        q_norm = normalize_tla(q_orig)
        test_samples.append({'q_orig': q_orig, 'q_norm': q_norm,
                              'ref_de': ref_de, 'ref_en': ref_en})
    k_results = []
    for k in k_values:
        TOP_K_RAG = k
        bleu_scores, recall_scores = [], []
        for s in test_samples:
            rag    = hybrid_search(s['q_orig'], top_k=k)
            recall = calc_recall_at_k(s['ref_de'], rag, k_values=[k])
            german = _get_best_german_from_rag(rag) if rag else ''
            en_out = german_to_english_strict(german) if german else ''
            bleu_scores.append(calc_bleu(s['ref_en'], en_out))
            recall_scores.append(recall.get(f'recall@{k}', 0.0))
        avg_bleu   = np.mean(bleu_scores)
        avg_recall = np.mean(recall_scores)
        composite  = 0.5 * avg_bleu + 0.5 * avg_recall
        k_results.append({'k': k, 'bleu': avg_bleu,
                           f'recall@{k}': avg_recall, 'composite': composite})
        print(f'  K={k:3d} | BLEU={avg_bleu:5.2f} | Recall={avg_recall:5.2f} | Composite={composite:5.2f}')
    df_k     = pd.DataFrame(k_results)
    best_row = df_k.loc[df_k['composite'].idxmax()]
    TOP_K_RAG = int(best_row['k'])
    print(f'\n🏆 Optimal K = {TOP_K_RAG} (composite score = {best_row["composite"]:.2f})')
    df_k.to_csv(os.path.join(_WORK_ROOT, 'k_optimization_results.csv'), index=False)
    return df_k

# df_k_results = run_k_optimization(num_samples=10)


## CELL 16 - Kill Old Flask Process

In [ ]:
import os
os.system('kill -9 $(lsof -t -i:5006) 2>/dev/null')
time.sleep(2)
print('✅ Port 5006 cleared')


## CELL 17 - Flask API + Ngrok v14

In [19]:
import threading
from pyngrok import ngrok, conf
from flask import Flask, request, jsonify
from flask_cors import CORS

ngrok.kill(); time.sleep(2)

# FIX G: Load token from environment variable — NEVER hardcode in source!
NGROK_TOKEN = os.environ.get('NGROK_AUTH_TOKEN', '')
if not NGROK_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        NGROK_TOKEN = UserSecretsClient().get_secret("NGROK_AUTH_TOKEN")
    except Exception:
        pass
if not NGROK_TOKEN:
    print('⚠️  NGROK_AUTH_TOKEN not found — set it with:')
    print('   import os; os.environ["NGROK_AUTH_TOKEN"] = "your_token"')
conf.get_default().auth_token = NGROK_TOKEN

app = Flask(__name__)
CORS(app)

ERROR_CODES = {
    'MISSING_INPUT' : ('400', 'Required field is missing'),
    'INVALID_INPUT' : ('400', 'Input is malformed or empty'),
    'PIPELINE_ERROR': ('500', 'Internal pipeline failure'),
    'MODEL_ERROR'   : ('503', 'Language model unavailable'),
}

def _api_error(code_key: str, detail: str = '', status: int = None):
    code, msg = ERROR_CODES.get(code_key, ('500', 'Unknown error'))
    return jsonify({'success': False,
                    'error': {'code': code_key, 'message': msg, 'detail': detail}}), status or int(code)

def _run_pipeline_safe(gardiner_str: str):
    last_exc = None
    for attempt in range(2):
        try:
            return full_pipeline(gardiner_str, verbose=False), None
        except RuntimeError as e:
            last_exc = e; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        except Exception as e:
            return None, str(e)
    return None, str(last_exc)


@app.route('/api/decipher', methods=['POST'])
def decipher():
    data  = request.get_json(silent=True)
    if not data: return _api_error('MISSING_INPUT', 'JSON body required')
    codes = data.get('codes', [])
    if not codes: return _api_error('MISSING_INPUT', '"codes" list is required')
    if not isinstance(codes, list) or not all(isinstance(c, str) for c in codes):
        return _api_error('INVALID_INPUT', '"codes" must be a list of strings')
    gardiner_str = ' '.join(codes).upper()
    result, err  = _run_pipeline_safe(gardiner_str)
    if err: return _api_error('PIPELINE_ERROR', err)
    code_list  = gardiner_str.split()
    tla_tokens = result['tla_phonemes'].split()
    meanings   = result['meanings']
    per_sign   = []
    for i, c in enumerate(code_list):
        key  = c.strip().lower()
        info = GARDINER_MAP.get(key, {})
        per_sign.append([c, info.get('unicode', ''),
                          tla_tokens[i] if i < len(tla_tokens) else '',
                          meanings[i]   if i < len(meanings)   else ''])
    glyphs_str = ''.join(GARDINER_MAP.get(c.strip().lower(), {}).get('unicode', '')
                          for c in code_list)
    intent = result.get('intention', {})
    sent   = result.get('sentiment', {})
    rag_s  = result['rag_score']
    return jsonify({'success': True, 'data': {
        'path'                    : result['path'],
        'glyphs'                  : glyphs_str,
        'per_sign'                : per_sign,
        'tla_norm'                : result['tla_normalised'],
        'english'                 : result['english'],
        'arabic'                  : result['arabic'],
        'arabic_verified'         : result.get('arabic_verified', None),
        'arabic_confidence'       : result.get('arabic_confidence', None),
        'sentiment'               : sent.get('label', 'neutral').lower(),
        'sent_score'              : f"{sent.get('confidence', 0):.0%}",
        'intention_en'            : intent.get('intention_en', ''),
        'intention_ar'            : intent.get('intention_ar', ''),
        'intent_confidence'       : intent.get('confidence', None),
        'rag_score'               : rag_s,
        'rag_score_max'           : RRF_SCORE_MAX,
        'rag_score_interpretation': interpret_rag_score(rag_s),
        'assembled_words'         : result.get('assembled_words', []),
        'assembly_metadata'       : result.get('assembly_metadata', []),
        'assembly_alternatives'   : result.get('assembly_alts', []),
        'assembly_score'          : result.get('assembly_score', None),
        'german_source'           : result.get('german_source', ''),
        'time'                    : result['processing_time'],
    }}), 200


@app.route('/api/translate', methods=['POST'])
def api_translate():
    data     = request.get_json(silent=True)
    if not data: return _api_error('MISSING_INPUT', 'JSON body required')
    gardiner = data.get('gardiner', '').strip()
    if not gardiner: return _api_error('MISSING_INPUT', '"gardiner" field is required')
    result, err = _run_pipeline_safe(gardiner)
    if err: return _api_error('PIPELINE_ERROR', err)
    rag_s = result['rag_score']
    return jsonify({'success': True,
        'input'                   : result['input'],
        'path'                    : result['path'],
        'tla_phonemes'            : result['tla_phonemes'],
        'tla_normalised'          : result['tla_normalised'],
        'meanings'                : result['meanings'],
        'english'                 : result['english'],
        'arabic'                  : result['arabic'],
        'arabic_verified'         : result.get('arabic_verified'),
        'sentiment'               : result.get('sentiment'),
        'intention'               : result.get('intention'),
        'rag_score'               : rag_s,
        'rag_score_max'           : RRF_SCORE_MAX,
        'rag_score_interpretation': interpret_rag_score(rag_s),
        'rag_results'             : result.get('rag_results', [])[:3],
        'german_source'           : result.get('german_source', ''),
        'time'                    : result['processing_time'],
    }), 200


@app.route('/api/evaluate', methods=['POST'])
def api_evaluate():
    data     = request.get_json(silent=True)
    if not data: return _api_error('MISSING_INPUT', 'JSON body required')
    gardiner = data.get('gardiner', '').strip()
    if not gardiner: return _api_error('MISSING_INPUT', '"gardiner" field is required')
    ref_en = data.get('reference_english', '')
    ref_de = data.get('reference_german',  '')
    try:
        result = evaluate_pipeline(gardiner, reference_english=ref_en,
                                   reference_german=ref_de)
    except Exception as e:
        return _api_error('PIPELINE_ERROR', str(e))
    eval_metrics = {k: result[k] for k in [
        'bleu','rouge1','rouge2','rougeL','meteor','chrf',
        'exact_match','word_overlap',
        'recall@1','recall@3','recall@5','recall@10','recall@20',
        'mrr','avg_retrieval_score',
    ] if k in result}
    return jsonify({'success': True, 'input': gardiner,
                    'english': result['english'], 'metrics': eval_metrics}), 200


@app.route('/api/status', methods=['GET'])
def status():
    return jsonify({
        'status'                 : 'ready',
        'version'                : 'v14-Graduation-Final',
        'environment'            : ENV,
        'model'                  : 'Seed-X-PPO-7B',
        'signs_loaded'           : len(GARDINER_MAP),
        'faiss_vectors'          : faiss_index.ntotal if faiss_index else 0,
        'test_set_rows'          : len(df_test_global) if df_test_global is not None else 0,
        'intention_classes'      : len(INTENTION_MAP),
        'arabic_verification'    : True,
        'prompt_token_limit'     : MAX_PROMPT_TOKENS,
        'rag_k'                  : RRF_K,
        'rag_alpha'              : HYBRID_ALPHA,
        'rag_score_max'          : RRF_SCORE_MAX,
        'similarity_threshold'   : SIMILARITY_THRESHOLD,
    }), 200

@app.route('/api/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'version': 'v14-Graduation-Final'}), 200

@app.route('/api/examples', methods=['GET'])
def examples():
    return jsonify({
        'simple_word'   : {'codes': ['O1'],                'description': 'House'},
        'monument'      : {'codes': ['G17', 'N35', 'D21'], 'description': 'Monument'},
        'sun_disk'      : {'codes': ['N5'],                'description': 'Sun / Ra'},
        'son_of_ra'     : {'codes': ['O4', 'N5'],          'description': 'Son of Ra'},
        'royal_offering': {'codes': ['R4', 'X8', 'A42'],   'description': 'Offering of the king'},
        'horus_injury'  : {'codes': ['S29','M17','G1','V13','D57','N35','S29','M17','D36','V13','D57','G5'],
                           'description': 'Injury for the one who injures Horus'},
    }), 200


def run_server():
    app.run(host='0.0.0.0', port=5010, use_reloader=False, debug=False)

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

tunnel     = ngrok.connect(5010)
PUBLIC_URL = tunnel.public_url

print('=' * 65)
print('🏛️  HIEROGLYPH NLP PIPELINE v14 — GRADUATION FINAL')
print(f'🌐  Public URL:\n\n    {PUBLIC_URL}\n')
print('📡  Endpoints:')
print(f'      POST {PUBLIC_URL}/api/decipher')
print(f'      POST {PUBLIC_URL}/api/translate')
print(f'      POST {PUBLIC_URL}/api/evaluate')
print(f'      GET  {PUBLIC_URL}/api/status')
print(f'      GET  {PUBLIC_URL}/api/health')
print(f'      GET  {PUBLIC_URL}/api/examples')
print('=' * 65)


 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5010 is in use by another program. Either identify and stop that program, or start the server with a different port.


🏛️  HIEROGLYPH NLP PIPELINE v14 — GRADUATION FINAL
🌐  Public URL:

    https://irretrievably-unsimpering-darrin.ngrok-free.dev

📡  Endpoints:
      POST https://irretrievably-unsimpering-darrin.ngrok-free.dev/api/decipher
      POST https://irretrievably-unsimpering-darrin.ngrok-free.dev/api/translate
      POST https://irretrievably-unsimpering-darrin.ngrok-free.dev/api/evaluate
      GET  https://irretrievably-unsimpering-darrin.ngrok-free.dev/api/status
      GET  https://irretrievably-unsimpering-darrin.ngrok-free.dev/api/health
      GET  https://irretrievably-unsimpering-darrin.ngrok-free.dev/api/examples


127.0.0.1 - - [09/Apr/2026 02:36:38] "OPTIONS /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 02:36:39] "GET /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 02:36:42] "OPTIONS /api/decipher HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 02:37:21] "POST /api/decipher HTTP/1.1" 200 -
